# Handwash Training Pipeline (Custom Annotated Dataset)

This notebook is adapted to train **MobileNetV2** on **your own annotated dataset** laid out like this:

```text
dataset_root/
├── No Action/
├── Soaping/
├── Step_1/
├── Step_2/
├── Step_3/
├── Step_4/
├── Step_5/
└── Step_6/
```

It assumes each class folder contains image files. The output classes are exactly the same as the folder names above.


In [ ]:
# Install dependencies
!pip install -q --no-cache-dir scikit-learn pandas numpy matplotlib seaborn opencv-python-headless


In [ ]:
import os
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

if tf.config.list_physical_devices("GPU"):
    for gpu in tf.config.list_physical_devices("GPU"):
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass
    print("GPU is available. Training will run on GPU.")
else:
    print("WARNING: No GPU detected. In Kaggle/Colab, enable GPU from the notebook settings.")


## Configuration

Set `DATASET_ROOT` to the folder that contains the eight class subfolders.


In [ ]:
# User config
DATASET_ROOT = Path("/kaggle/input/your-handwash-dataset")  # <- change this
RUN_NAME = os.environ.get("RUN_NAME", "handwash_custom_mobilenetv2")

CLASS_NAMES = [
    "No Action",
    "Soaping",
    "Step_1",
    "Step_2",
    "Step_3",
    "Step_4",
    "Step_5",
    "Step_6",
]
NUM_CLASSES = len(CLASS_NAMES)

IMG_SIZE = (224, 224)
BATCH_SIZE = 64
EPOCHS = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.0

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

USE_MIXED_PRECISION = True
USE_IMAGENET_WEIGHTS = True
FREEZE_BASE_EPOCHS = 5
FINE_TUNE_EPOCHS = 15
FINE_TUNE_AT = 100  # unfreeze deeper MobileNetV2 layers after warmup

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

assert math.isclose(TRAIN_RATIO + VAL_RATIO + TEST_RATIO, 1.0, rel_tol=1e-6), "Split ratios must add to 1.0"

WORK_DIR = Path("/kaggle/working") / RUN_NAME
MODEL_DIR = WORK_DIR / "models"
ARTIFACT_DIR = WORK_DIR / "artifacts"
for p in [WORK_DIR, MODEL_DIR, ARTIFACT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if USE_MIXED_PRECISION and tf.config.list_physical_devices("GPU"):
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision enabled:", mixed_precision.global_policy())


In [ ]:
# Optional multi-GPU support
gpus = tf.config.list_physical_devices("GPU")
if len(gpus) > 1:
    STRATEGY = tf.distribute.MirroredStrategy()
else:
    STRATEGY = tf.distribute.get_strategy()

print("Using strategy:", type(STRATEGY).__name__, "| replicas:", STRATEGY.num_replicas_in_sync)


In [ ]:
def index_dataset(dataset_root: Path, class_names):
    dataset_root = Path(dataset_root)
    if not dataset_root.exists():
        raise FileNotFoundError(f"Dataset root not found: {dataset_root}")

    rows = []
    for class_name in class_names:
        class_dir = dataset_root / class_name
        if not class_dir.exists():
            raise FileNotFoundError(f"Missing class folder: {class_dir}")
        for path in class_dir.rglob("*"):
            if path.is_file() and path.suffix.lower() in IMAGE_EXTS:
                rows.append({
                    "filepath": str(path),
                    "class_name": class_name,
                    "class_id": class_names.index(class_name),
                })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"No image files found under {dataset_root}")

    return df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)


df = index_dataset(DATASET_ROOT, CLASS_NAMES)
print("Total images:", len(df))
display(df.head())

print("\nClass distribution:")
print(df["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0).to_string())


In [ ]:
# Stratified train/val/test split
train_df, temp_df = train_test_split(
    df,
    test_size=(1.0 - TRAIN_RATIO),
    stratify=df["class_id"],
    random_state=SEED,
)

val_fraction_of_temp = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
val_df, test_df = train_test_split(
    temp_df,
    test_size=(1.0 - val_fraction_of_temp),
    stratify=temp_df["class_id"],
    random_state=SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Val:  ", len(val_df))
print("Test: ", len(test_df))


In [ ]:
def plot_split_distribution(train_df, val_df, test_df):
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    splits = [("Train", train_df), ("Val", val_df), ("Test", test_df)]
    for ax, (name, split_df) in zip(axes, splits):
        counts = split_df["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0)
        counts.plot(kind="bar", ax=ax)
        ax.set_title(f"{name} distribution")
        ax.set_xlabel("Class")
        ax.set_ylabel("Count")
        ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()

plot_split_distribution(train_df, val_df, test_df)


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def decode_and_resize(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image, tf.one_hot(label, NUM_CLASSES)

def augment_image(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.15)
    image = tf.image.random_contrast(image, lower=0.85, upper=1.15)
    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, label

def build_dataset(split_df, training=False):
    ds = tf.data.Dataset.from_tensor_slices((
        split_df["filepath"].values,
        split_df["class_id"].values.astype("int32"),
    ))
    if training:
        ds = ds.shuffle(len(split_df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_and_resize, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(augment_image, num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (keras.applications.mobilenet_v2.preprocess_input(x), y),
                num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = build_dataset(train_df, training=True)
val_ds = build_dataset(val_df, training=False)
test_ds = build_dataset(test_df, training=False)


In [ ]:
def show_batch(ds, class_names):
    images, labels = next(iter(ds.take(1)))
    plt.figure(figsize=(12, 8))
    for i in range(min(12, images.shape[0])):
        plt.subplot(3, 4, i + 1)
        img = images[i].numpy()
        # Reverse preprocess_input approximately for visualization
        img = ((img + 1.0) * 127.5).clip(0, 255).astype("uint8")
        plt.imshow(img)
        plt.title(class_names[int(tf.argmax(labels[i]).numpy())], fontsize=9)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_batch(train_ds, CLASS_NAMES)


In [ ]:
def build_model():
    inputs = keras.Input(shape=(*IMG_SIZE, 3), name="image")
    base_model = keras.applications.MobileNetV2(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights="imagenet" if USE_IMAGENET_WEIGHTS else None,
    )
    base_model.trainable = False

    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)

    model = keras.Model(inputs, outputs, name="handwash_mobilenetv2")
    return model, base_model

def compile_model(model, lr):
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=WEIGHT_DECAY),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=[
            "accuracy",
            keras.metrics.TopKCategoricalAccuracy(k=2, name="top2_accuracy"),
        ],
    )


In [ ]:
with STRATEGY.scope():
    model, base_model = build_model()
    compile_model(model, LEARNING_RATE)

model.summary()


In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        str(MODEL_DIR / "best_mobilenetv2.keras"),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        patience=2,
        factor=0.5,
        min_lr=1e-6,
    ),
    keras.callbacks.CSVLogger(str(ARTIFACT_DIR / "training_log.csv")),
]

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FREEZE_BASE_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
# Fine-tune MobileNetV2
base_model.trainable = True

for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

compile_model(model, LEARNING_RATE * 0.1)

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=history_stage1.epoch[-1] + 1 if history_stage1.epoch else 0,
    epochs=FREEZE_BASE_EPOCHS + FINE_TUNE_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
def plot_history(*histories):
    merged = {}
    for history in histories:
        for k, v in history.history.items():
            merged.setdefault(k, []).extend(v)

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(merged.get("loss", []), label="train_loss")
    plt.plot(merged.get("val_loss", []), label="val_loss")
    plt.legend()
    plt.title("Loss")

    plt.subplot(1, 2, 2)
    plt.plot(merged.get("accuracy", []), label="train_acc")
    plt.plot(merged.get("val_accuracy", []), label="val_acc")
    plt.legend()
    plt.title("Accuracy")

    plt.tight_layout()
    plt.show()

plot_history(history_stage1, history_stage2)


In [ ]:
# Evaluate on the test set
test_metrics = model.evaluate(test_ds, verbose=1)
print(dict(zip(model.metrics_names, test_metrics)))


In [ ]:
# Predictions + confusion matrix
y_true = test_df["class_id"].to_numpy()
pred_probs = model.predict(test_ds, verbose=1)
y_pred = np.argmax(pred_probs, axis=1)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
# Save final model and label mapping
final_model_path = MODEL_DIR / "mobilenetv2_handwash_custom_final.keras"
model.save(final_model_path)

label_map = {i: name for i, name in enumerate(CLASS_NAMES)}
with open(ARTIFACT_DIR / "label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)

split_info = {
    "train_size": int(len(train_df)),
    "val_size": int(len(val_df)),
    "test_size": int(len(test_df)),
    "class_names": CLASS_NAMES,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
}
with open(ARTIFACT_DIR / "run_config.json", "w") as f:
    json.dump(split_info, f, indent=2)

print("Saved model to:", final_model_path)
print("Saved label map to:", ARTIFACT_DIR / "label_map.json")
print("Saved run config to:", ARTIFACT_DIR / "run_config.json")


## Notes

- This version no longer downloads or mixes the old Kaggle / PSKUS / METC / synthetic datasets.
- It uses **only your custom annotated dataset**.
- The model is fixed to **MobileNetV2**.
- The output classes are fixed to:

```python
["No Action", "Soaping", "Step_1", "Step_2", "Step_3", "Step_4", "Step_5", "Step_6"]
```

- Training will run on **GPU** when the notebook runtime has GPU enabled.
